In [ ]:
# Does SPSA+GPU work with batched_shots_gpu=False? Same pinned versions, only that toggled.

!pip install -q qiskit==1.4.6 qiskit-aer-gpu==0.15.1 qiskit-algorithms==0.4.0 qiskit-optimization==0.7.0 qiskit-ibm-runtime==0.29.0 2>&1 | tail -5

import numpy as np
from qiskit.circuit.library import TwoLocal, PauliTwoDesign
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeCairoV2
from qiskit_algorithms import SamplingVQE
from qiskit_algorithms.optimizers import SPSA, COBYLA
from qiskit.quantum_info import SparsePauliOp
from qiskit import transpile

import qiskit, qiskit_aer
print("qiskit:", qiskit.__version__)
print("qiskit_aer:", qiskit_aer.__version__)
from qiskit_aer import AerSimulator
print("available devices:", AerSimulator().available_devices())

# 15 qubits, diagonal Z + ZZ like the real problem, but cheap enough to try several configs
rng = np.random.default_rng(0)
n_qubits = 15
terms = []
for i in range(n_qubits):
    lab = ["I"] * n_qubits; lab[i] = "Z"
    terms.append(("".join(lab), float(rng.uniform(-2, 2))))
for i in range(n_qubits - 1):
    lab = ["I"] * n_qubits; lab[i] = "Z"; lab[i+1] = "Z"
    terms.append(("".join(lab), float(rng.uniform(-2, 2))))
ising_op = SparsePauliOp.from_list(terms)

cairo_nm = NoiseModel.from_backend(FakeCairoV2())
print("noise model error channels:", len(cairo_nm.to_dict()["errors"]))

def run_case(label, batched_shots_gpu, maxiter=15):
    print(f"\n--- {label} (batched_shots_gpu={batched_shots_gpu}) ---")
    sampler = AerSampler(
        seed=42,
        options={"backend_options": {
            "noise_model": cairo_nm, "method": "statevector", "device": "GPU",
            "batched_shots_gpu": batched_shots_gpu, "batched_shots_gpu_max_qubits": 16,
            "max_parallel_threads": 0, "max_parallel_experiments": 0,
        }},
    )
    sampler.options.default_shots = 2000
    # RealAmplit-circular with SPSA was the original failure case
    ansatz = TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz",
                       entanglement="circular", reps=3)
    ansatz_d = transpile(ansatz.decompose(reps=10), backend=sampler._backend, optimization_level=1)
    init = np.random.default_rng(1).uniform(-np.pi, np.pi, size=ansatz_d.num_parameters)
    try:
        vqe = SamplingVQE(sampler=sampler, ansatz=ansatz_d, optimizer=SPSA(maxiter=maxiter), initial_point=init)
        result = vqe.compute_minimum_eigenvalue(ising_op)
        print(f"  SUCCESS: eigenvalue={float(np.real(result.eigenvalue)):.4f}")
        return True
    except Exception as exc:
        print(f"  FAILED: {type(exc).__name__}: {exc}")
        return False

r1 = run_case("batched ON", True)     # should reproduce the original failure
r2 = run_case("batched OFF", False)   # the actual diagnostic

print(f"\nbatched_shots_gpu=True  -> {'worked' if r1 else 'failed'}")
print(f"batched_shots_gpu=False -> {'worked' if r2 else 'failed'}")